In [1]:
import requests

for url in [
    "https://dummyjson.com/products?limit=3",
    "https://dummyjson.com/carts?limit=3",
    "https://dummyjson.com/users?limit=3"
]:
    r = requests.get(url)
    data = r.json()
    top_key = list(data.keys())
    print(f"URL: {url}")
    print(f"  Status: {r.status_code}")
    print(f"  Top-level keys: {top_key}")
    print(f"  Record count in response: {len(data[top_key[0]])}")
    print()

URL: https://dummyjson.com/products?limit=3
  Status: 200
  Top-level keys: ['products', 'total', 'skip', 'limit']
  Record count in response: 3

URL: https://dummyjson.com/carts?limit=3
  Status: 200
  Top-level keys: ['carts', 'total', 'skip', 'limit']
  Record count in response: 3

URL: https://dummyjson.com/users?limit=3
  Status: 200
  Top-level keys: ['users', 'total', 'skip', 'limit']
  Record count in response: 3



In [2]:
import requests

r = requests.get("https://dummyjson.com/products?limit=3")
data = r.json()

# Loop through and print each product
for product in data['products']:
    print(f"ID:       {product['id']}")
    print(f"Title:    {product['title']}")
    print(f"Price:    {product['price']}")
    print(f"Category: {product['category']}")
    print(f"Brand:    {product.get('brand', 'N/A')}")
    print(f"Stock:    {product['stock']}")
    print("---")

ID:       1
Title:    Essence Mascara Lash Princess
Price:    9.99
Category: beauty
Brand:    Essence
Stock:    99
---
ID:       2
Title:    Eyeshadow Palette with Mirror
Price:    19.99
Category: beauty
Brand:    Glamour Beauty
Stock:    34
---
ID:       3
Title:    Powder Canister
Price:    14.99
Category: beauty
Brand:    Velvet Touch
Stock:    89
---


In [3]:
# Make sure your api_client.py is in the same folder as test.ipynb
from api_client import fetch_products, fetch_carts, fetch_users

# ── Test Products ────────────────────────────────────────────────────────────
products_raw = fetch_products()
print(type(products_raw))          # should be: <class 'list'>
print(len(products_raw))           # should be: 100
print(products_raw[0].keys())      # should show: id, title, price, etc.
print("---")

# ── Test Carts ───────────────────────────────────────────────────────────────
carts_raw = fetch_carts()
print(type(carts_raw))             # should be: <class 'list'>
print(len(carts_raw))              # should be: 100
print(carts_raw[0].keys())         # should show: id, products, total, userId etc.
print("---")

# ── Test Users ───────────────────────────────────────────────────────────────
users_raw = fetch_users()
print(type(users_raw))             # should be: <class 'list'>
print(len(users_raw))              # should be: 100
print(users_raw[0].keys())         # should show: id, firstName, lastName etc.

SUCCESS: Fetched 100 records from https://dummyjson.com/products?limit=100
<class 'list'>
100
dict_keys(['id', 'title', 'description', 'category', 'price', 'discountPercentage', 'rating', 'stock', 'tags', 'brand', 'sku', 'weight', 'dimensions', 'warrantyInformation', 'shippingInformation', 'availabilityStatus', 'reviews', 'returnPolicy', 'minimumOrderQuantity', 'meta', 'images', 'thumbnail'])
---
SUCCESS: Fetched 100 records from https://dummyjson.com/carts?limit=100
<class 'list'>
100
dict_keys(['id', 'products', 'total', 'discountedTotal', 'userId', 'totalProducts', 'totalQuantity'])
---
SUCCESS: Fetched 100 records from https://dummyjson.com/users?limit=100
<class 'list'>
100
dict_keys(['id', 'firstName', 'lastName', 'maidenName', 'age', 'gender', 'email', 'phone', 'username', 'password', 'birthDate', 'image', 'bloodGroup', 'height', 'weight', 'eyeColor', 'hair', 'ip', 'address', 'macAddress', 'university', 'bank', 'company', 'ein', 'ssn', 'userAgent', 'crypto', 'role'])


In [4]:
from transformer import transform_products, transform_carts, transform_users
from api_client   import fetch_products, fetch_carts, fetch_users

# ── Fetch raw data first ──────────────────────────────────────────────────────
products_raw = fetch_products()
carts_raw    = fetch_carts()
users_raw    = fetch_users()

# ── Transform ─────────────────────────────────────────────────────────────────
df_products, df_reviews  = transform_products(products_raw)
df_carts, df_cart_items  = transform_carts(carts_raw)
df_users                 = transform_users(users_raw)

# ── Verify products ───────────────────────────────────────────────────────────
print("=== PRODUCTS ===")
print(df_products.shape)               # should be (100, 23 columns)
print(df_products.head(3))             # first 3 rows as a table
print(df_products.dtypes)              # column data types
print("---")

# ── Verify reviews ────────────────────────────────────────────────────────────
print("=== REVIEWS ===")
print(df_reviews.shape)                # should be (300, 6 columns)
print(df_reviews.head(3))              # first 3 rows
print("---")

# ── Verify carts ──────────────────────────────────────────────────────────────
print("=== CARTS ===")
print(df_carts.shape)                  # should be (100, 6 columns)
print(df_carts.head(3))                # first 3 rows
print("---")

# ── Verify cart items ─────────────────────────────────────────────────────────
print("=== CART ITEMS ===")
print(df_cart_items.shape)             # should be (400+, 9 columns)
print(df_cart_items.head(3))           # first 3 rows
print("---")

# ── Verify users ──────────────────────────────────────────────────────────────
print("=== USERS ===")
print(df_users.shape)                  # should be (100, 29 columns)
print(df_users.head(3))                # first 3 rows
print(df_users.columns.tolist())       # full list of column names

# ── Confirm sensitive fields were dropped ─────────────────────────────────────
print("---")
print("=== SENSITIVE FIELD CHECK ===")
sensitive = ["password", "ssn", "ein", "ip", "macAddress", "userAgent", "crypto"]
for field in sensitive:
    print(f"{field} in df_users: {field in df_users.columns}")  # all should be False

SUCCESS: Fetched 100 records from https://dummyjson.com/products?limit=100
SUCCESS: Fetched 100 records from https://dummyjson.com/carts?limit=100
SUCCESS: Fetched 100 records from https://dummyjson.com/users?limit=100
SUCCESS: Transformed products  - 100 rows, 300 reviews
SUCCESS: Transformed carts     - 100 rows, 378 line items
SUCCESS: Transformed users     - 100 rows
=== PRODUCTS ===
(100, 23)
   id                          title  \
0   1  Essence Mascara Lash Princess   
1   2  Eyeshadow Palette with Mirror   
2   3                Powder Canister   

                                         description category  price  \
0  The Essence Mascara Lash Princess is a popular...   beauty   9.99   
1  The Eyeshadow Palette with Mirror offers a ver...   beauty  19.99   
2  The Powder Canister is a finely milled setting...   beauty  14.99   

   discountPercentage  rating  stock           brand              sku  ...  \
0               10.48    2.56     99         Essence  BEA-ESS-ESS-001  

In [5]:
from db_connector import get_connection

# Try to connect
conn = get_connection()

# Check if it worked
if conn:
    print("SUCCESS: Connection established!")
    print(f"Connection type: {type(conn)}")
    conn.close()
else:
    print("ERROR: Connection failed")

SUCCESS: Connected to localhost\SQLEXPRESS - EcommerceDB
SUCCESS: Connection established!
Connection type: <class 'pyodbc.Connection'>


In [6]:
from dotenv import load_dotenv
import os

load_dotenv()

server   = os.getenv("SQL_SERVER")
database = os.getenv("SQL_DATABASE")

print(f"Server   : '{server}'")
print(f"Database : '{database}'")

Server   : 'localhost\SQLEXPRESS'
Database : 'EcommerceDB'


In [7]:
import pyodbc

# Hardcoded connection string - just for testing
conn = pyodbc.connect(
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=.\\SQLEXPRESS;"
    "Database=master;"
    "Trusted_Connection=yes;"
)

print("Connection successful!")
conn.close()

Connection successful!


In [8]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

server   = os.getenv("SQL_SERVER")
database = os.getenv("SQL_DATABASE")
username = os.getenv("SQL_USERNAME")

print(f"Server   : '{server}'")
print(f"Database : '{database}'")
print(f"Username : '{username}'")

Server   : 'localhost\SQLEXPRESS'
Database : 'EcommerceDB'
Username : 'JMoore\jimmi'


In [9]:
from db_connector import get_connection

conn = get_connection()

if conn:
    print("Module 3 complete!")
    conn.close()
else:
    print("Connection failed")

SUCCESS: Connected to localhost\SQLEXPRESS - EcommerceDB
Module 3 complete!


In [17]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

server   = os.getenv("SQL_SERVER")
database = os.getenv("SQL_DATABASE")

print(f"Server   : '{server}'")
print(f"Database : '{database}'")

Server   : '.\\SQLEXPRESS'
Database : 'EcommerceDB'


In [19]:
from db_connector import get_connection

conn = get_connection()

if conn:
    print("Module 3 complete!")
    conn.close()
else:
    print("Connection failed")

ERROR: Could not connect to SQL Server - ('08001', '[08001] [Microsoft][ODBC Driver 17 for SQL Server]Named Pipes Provider: Could not open a connection to SQL Server [2].  (2) (SQLDriverConnect); [08001] [Microsoft][ODBC Driver 17 for SQL Server]Login timeout expired (0); [08001] [Microsoft][ODBC Driver 17 for SQL Server]A network-related or instance-specific error has occurred while establishing a connection to SQL Server. Server is not found or not accessible. Check if instance name is correct and if SQL Server is configured to allow remote connections. For more information see SQL Server Books Online. (2)')
Connection failed


In [20]:
import importlib
import db_connector
importlib.reload(db_connector)

from db_connector import get_connection

conn = get_connection()

if conn:
    print("Module 3 complete!")
    conn.close()
else:
    print("Connection failed")

ERROR: Could not connect to SQL Server - ('08001', '[08001] [Microsoft][ODBC Driver 17 for SQL Server]The client cannot connect to the server because the requested instance was not available. Use SQL Server Configuration Manager to make sure the SQL Server instance is configured correctly.  (0) (SQLDriverConnect)')
Connection failed


In [10]:
import pyodbc

# Test using localhost\\SQLEXPRESS
try:
    conn = pyodbc.connect(
        "Driver={ODBC Driver 17 for SQL Server};"
        "Server=localhost\\SQLEXPRESS;"
        "Database=master;"
        "Trusted_Connection=yes;"
    )
    print("SUCCESS: localhost\\SQLEXPRESS works!")
    conn.close()
except Exception as e:
    print(f"FAILED: {e}")

SUCCESS: localhost\SQLEXPRESS works!


In [22]:
import importlib
import db_connector
importlib.reload(db_connector)

from db_connector import get_connection

conn = get_connection()

if conn:
    print("Module 3 complete!")
    conn.close()
else:
    print("Connection failed")

ERROR: Could not connect to SQL Server - ('08001', '[08001] [Microsoft][ODBC Driver 17 for SQL Server]The client cannot connect to the server because the requested instance was not available. Use SQL Server Configuration Manager to make sure the SQL Server instance is configured correctly.  (0) (SQLDriverConnect)')
Connection failed


In [23]:
import importlib
import db_connector
importlib.reload(db_connector)

from db_connector import get_connection

conn = get_connection()

if conn:
    print("Module 3 complete!")
    conn.close()
else:
    print("Connection failed")

ERROR: Could not connect to SQL Server - ('08001', '[08001] [Microsoft][ODBC Driver 17 for SQL Server]The client cannot connect to the server because the requested instance was not available. Use SQL Server Configuration Manager to make sure the SQL Server instance is configured correctly.  (0) (SQLDriverConnect)')
Connection failed


In [24]:
import importlib
import db_connector
importlib.reload(db_connector)

from db_connector import get_connection

conn = get_connection()

if conn:
    print("Module 3 complete!")
    conn.close()
else:
    print("Connection failed")

ERROR: Could not connect to SQL Server - ('08001', '[08001] [Microsoft][ODBC Driver 17 for SQL Server]The client cannot connect to the server because the requested instance was not available. Use SQL Server Configuration Manager to make sure the SQL Server instance is configured correctly.  (0) (SQLDriverConnect)')
Connection failed


In [25]:
import importlib
import db_connector
importlib.reload(db_connector)

from db_connector import get_connection

conn = get_connection()

if conn:
    print("Module 3 complete!")
    conn.close()
else:
    print("Connection failed")

ERROR: Could not connect to SQL Server - ('08001', '[08001] [Microsoft][ODBC Driver 17 for SQL Server]The client cannot connect to the server because the requested instance was not available. Use SQL Server Configuration Manager to make sure the SQL Server instance is configured correctly.  (0) (SQLDriverConnect)')
Connection failed


In [11]:
import pyodbc

driver = "ODBC Driver 17 for SQL Server"

# List of every possible server name format to try
server_names = [
    "localhost\\SQLEXPRESS",
    ".\\SQLEXPRESS",
    "127.0.0.1\\SQLEXPRESS",
    "localhost",
    "(local)\\SQLEXPRESS",
]

for server in server_names:
    try:
        conn = pyodbc.connect(
            f"Driver={{{driver}}};"
            f"Server={server};"
            f"Database=master;"
            f"Trusted_Connection=yes;"
            f"Connect Timeout=5;"
        )
        print(f"SUCCESS: '{server}' works!")
        conn.close()
    except Exception as e:
        print(f"FAILED:  '{server}'")

SUCCESS: 'localhost\SQLEXPRESS' works!
SUCCESS: '.\SQLEXPRESS' works!
FAILED:  '127.0.0.1\SQLEXPRESS'
FAILED:  'localhost'
SUCCESS: '(local)\SQLEXPRESS' works!


In [27]:
from dotenv import load_dotenv
import os
import pyodbc

load_dotenv(override=True)

server = os.getenv("SQL_SERVER")
print(f"Server from .env: '{server}'")

try:
    conn = pyodbc.connect(
        f"Driver={{ODBC Driver 17 for SQL Server}};"
        f"Server={server};"
        f"Database=master;"
        f"Trusted_Connection=yes;"
    )
    print("SUCCESS: Connected!")
    conn.close()
except Exception as e:
    print(f"FAILED: {e}")

Server from .env: 'localhost/SQLEXPRESS'
FAILED: ('08001', '[08001] [Microsoft][ODBC Driver 17 for SQL Server]Named Pipes Provider: Could not open a connection to SQL Server [67].  (67) (SQLDriverConnect); [08001] [Microsoft][ODBC Driver 17 for SQL Server]Login timeout expired (0); [08001] [Microsoft][ODBC Driver 17 for SQL Server]A network-related or instance-specific error has occurred while establishing a connection to SQL Server. Server is not found or not accessible. Check if instance name is correct and if SQL Server is configured to allow remote connections. For more information see SQL Server Books Online. (67)')


In [28]:
from dotenv import load_dotenv
import os
import pyodbc

load_dotenv(override=True)

server = os.getenv("SQL_SERVER")
print(f"Server from .env: '{server}'")

try:
    conn = pyodbc.connect(
        f"Driver={{ODBC Driver 17 for SQL Server}};"
        f"Server={server};"
        f"Database=master;"
        f"Trusted_Connection=yes;"
    )
    print("SUCCESS: Connected!")
    conn.close()
except Exception as e:
    print(f"FAILED: {e}")

Server from .env: 'localhost\SQLEXPRESS'
SUCCESS: Connected!


In [29]:
import importlib
import db_connector
importlib.reload(db_connector)

from db_connector import get_connection

conn = get_connection()

if conn:
    print("Module 3 complete!")
    conn.close()
else:
    print("Connection failed")

ERROR: Unexpected error - ('28000', '[28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Login failed for user \'JMoore\\jimmi\'. (18456) (SQLDriverConnect); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot open database "EcommerceDB" requested by the login. The login failed. (4060); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Login failed for user \'JMoore\\jimmi\'. (18456); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot open database "EcommerceDB" requested by the login. The login failed. (4060)')
Connection failed


In [30]:
import importlib
import db_connector
importlib.reload(db_connector)

from db_connector import get_connection

conn = get_connection()

if conn:
    print("Module 3 complete!")
    conn.close()
else:
    print("Connection failed")

SUCCESS: Connected to localhost\SQLEXPRESS - EcommerceDB
Module 3 complete!


In [12]:
from db_writer import create_tables, write_products, write_carts, write_users
from transformer import transform_products, transform_carts, transform_users
from api_client import fetch_products, fetch_carts, fetch_users

# Step 1 - Create all tables
print("=== STEP 1: CREATE TABLES ===")
create_tables()
print()

# Step 2 - Fetch raw data
print("=== STEP 2: FETCH DATA ===")
products_raw = fetch_products()
carts_raw    = fetch_carts()
users_raw    = fetch_users()
print()

# Step 3 - Transform into DataFrames
print("=== STEP 3: TRANSFORM DATA ===")
df_products, df_reviews  = transform_products(products_raw)
df_carts, df_cart_items  = transform_carts(carts_raw)
df_users                 = transform_users(users_raw)
print()

# Step 4 - Write to SQL Server
print("=== STEP 4: WRITE TO SQL SERVER ===")
write_products(df_products, df_reviews)
write_carts(df_carts, df_cart_items)
write_users(df_users)
print()

print("=== ALL STEPS COMPLETE ===")

=== STEP 1: CREATE TABLES ===
SUCCESS: Connected to localhost\SQLEXPRESS - EcommerceDB
SUCCESS: products table ready
SUCCESS: product_reviews table ready
SUCCESS: carts table ready
SUCCESS: cart_items table ready
SUCCESS: users table ready

=== STEP 2: FETCH DATA ===
SUCCESS: Fetched 100 records from https://dummyjson.com/products?limit=100
SUCCESS: Fetched 100 records from https://dummyjson.com/carts?limit=100
SUCCESS: Fetched 100 records from https://dummyjson.com/users?limit=100

=== STEP 3: TRANSFORM DATA ===
SUCCESS: Transformed products  - 100 rows, 300 reviews
SUCCESS: Transformed carts     - 100 rows, 378 line items
SUCCESS: Transformed users     - 100 rows

=== STEP 4: WRITE TO SQL SERVER ===
SUCCESS: Connected to localhost\SQLEXPRESS - EcommerceDB
ERROR: write_products failed - ('42S02', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid object name 'sqlite_master'. (208) (SQLExecDirectW); [42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]State

c:\AI Data Pipeline Project\db_writer.py:200: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_products.to_sql(
c:\AI Data Pipeline Project\db_writer.py:254: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_carts.to_sql(
c:\AI Data Pipeline Project\db_writer.py:306: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_users.to_sql(


In [13]:
# Install required packages
import subprocess
subprocess.run(["pip", "install", "sqlalchemy", "pyodbc"], check=True)
print("Installation complete!")

Installation complete!


In [14]:
import importlib
import db_connector, db_writer
importlib.reload(db_connector)
importlib.reload(db_writer)

from db_writer import create_tables, write_products, write_carts, write_users
from transformer import transform_products, transform_carts, transform_users
from api_client import fetch_products, fetch_carts, fetch_users

# Step 1 - Create tables
print("=== STEP 1: CREATE TABLES ===")
create_tables()
print()

# Step 2 - Fetch raw data
print("=== STEP 2: FETCH DATA ===")
products_raw = fetch_products()
carts_raw    = fetch_carts()
users_raw    = fetch_users()
print()

# Step 3 - Transform
print("=== STEP 3: TRANSFORM DATA ===")
df_products, df_reviews  = transform_products(products_raw)
df_carts, df_cart_items  = transform_carts(carts_raw)
df_users                 = transform_users(users_raw)
print()

# Step 4 - Write to SQL Server
print("=== STEP 4: WRITE TO SQL SERVER ===")
write_products(df_products, df_reviews)
write_carts(df_carts, df_cart_items)
write_users(df_users)
print()

print("=== ALL STEPS COMPLETE ===")

=== STEP 1: CREATE TABLES ===
SUCCESS: Connected to localhost\SQLEXPRESS - EcommerceDB
SUCCESS: products table ready
SUCCESS: product_reviews table ready
SUCCESS: carts table ready
SUCCESS: cart_items table ready
SUCCESS: users table ready

=== STEP 2: FETCH DATA ===
SUCCESS: Fetched 100 records from https://dummyjson.com/products?limit=100
SUCCESS: Fetched 100 records from https://dummyjson.com/carts?limit=100
SUCCESS: Fetched 100 records from https://dummyjson.com/users?limit=100

=== STEP 3: TRANSFORM DATA ===
SUCCESS: Transformed products  - 100 rows, 300 reviews
SUCCESS: Transformed carts     - 100 rows, 378 line items
SUCCESS: Transformed users     - 100 rows

=== STEP 4: WRITE TO SQL SERVER ===
SUCCESS: Connected to localhost\SQLEXPRESS - EcommerceDB
ERROR: write_products failed - ('42S02', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid object name 'sqlite_master'. (208) (SQLExecDirectW); [42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]State

In [17]:
from db_connector import get_engine

engine = get_engine()
print(type(engine))
print(engine)

ImportError: cannot import name 'get_engine' from 'db_connector' (c:\AI Data Pipeline Project\db_connector.py)

In [19]:
import importlib
import db_connector
importlib.reload(db_connector)

from db_connector import get_connection, get_engine

# Test pyodbc connection
conn = get_connection()
if conn:
    print("pyodbc connection works!")
    conn.close()

# Test SQLAlchemy engine
engine = get_engine()
if engine:
    print("SQLAlchemy engine works!")

SUCCESS: Connected to localhost\SQLEXPRESS - EcommerceDB
pyodbc connection works!
SUCCESS: SQLAlchemy engine connected to localhost\SQLEXPRESS - EcommerceDB
SQLAlchemy engine works!


In [20]:
import importlib
import db_connector, db_writer
importlib.reload(db_connector)
importlib.reload(db_writer)

from db_writer import create_tables, write_products, write_carts, write_users
from transformer import transform_products, transform_carts, transform_users
from api_client import fetch_products, fetch_carts, fetch_users

# Step 1 - Create tables
print("=== STEP 1: CREATE TABLES ===")
create_tables()
print()

# Step 2 - Fetch raw data
print("=== STEP 2: FETCH DATA ===")
products_raw = fetch_products()
carts_raw    = fetch_carts()
users_raw    = fetch_users()
print()

# Step 3 - Transform
print("=== STEP 3: TRANSFORM DATA ===")
df_products, df_reviews  = transform_products(products_raw)
df_carts, df_cart_items  = transform_carts(carts_raw)
df_users                 = transform_users(users_raw)
print()

# Step 4 - Write to SQL Server
print("=== STEP 4: WRITE TO SQL SERVER ===")
write_products(df_products, df_reviews)
write_carts(df_carts, df_cart_items)
write_users(df_users)
print()

print("=== ALL STEPS COMPLETE ===")

=== STEP 1: CREATE TABLES ===
SUCCESS: Connected to localhost\SQLEXPRESS - EcommerceDB
SUCCESS: products table ready
SUCCESS: product_reviews table ready
SUCCESS: carts table ready
SUCCESS: cart_items table ready
SUCCESS: users table ready

=== STEP 2: FETCH DATA ===
SUCCESS: Fetched 100 records from https://dummyjson.com/products?limit=100
SUCCESS: Fetched 100 records from https://dummyjson.com/carts?limit=100
SUCCESS: Fetched 100 records from https://dummyjson.com/users?limit=100

=== STEP 3: TRANSFORM DATA ===
SUCCESS: Transformed products  - 100 rows, 300 reviews
SUCCESS: Transformed carts     - 100 rows, 378 line items
SUCCESS: Transformed users     - 100 rows

=== STEP 4: WRITE TO SQL SERVER ===
SUCCESS: Connected to localhost\SQLEXPRESS - EcommerceDB
ERROR: write_products failed - ('42S02', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid object name 'sqlite_master'. (208) (SQLExecDirectW); [42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]State

In [21]:
import inspect
import db_writer
importlib.reload(db_writer)

# Print the write_products function source code
print(inspect.getsource(db_writer.write_products))

def write_products(df_products, df_reviews):
    """
    Deletes existing data and inserts fresh products
    and product_reviews DataFrames into SQL Server.

    Args:
        df_products : DataFrame with 100 products
        df_reviews  : DataFrame with 300 reviews
    """
    conn = get_connection()

    if conn is None:
        print("ERROR: write_products failed - no database connection")
        return False

    try:
        cursor = conn.cursor()

        # Delete existing data first - reviews before products
        # because reviews has a foreign key pointing to products
        cursor.execute("DELETE FROM product_reviews")
        cursor.execute("DELETE FROM products")
        conn.commit()
        cursor.close()

        # Insert products DataFrame into SQL Server
        df_products.to_sql(
            "products",
            conn,
            if_exists="append",   # append to existing table structure
            index=False,          # don't write DataFrame index as a col

In [22]:
import importlib
import db_connector, db_writer
importlib.reload(db_connector)
importlib.reload(db_writer)

from db_writer import create_tables, write_products, write_carts, write_users
from transformer import transform_products, transform_carts, transform_users
from api_client import fetch_products, fetch_carts, fetch_users

print("=== STEP 1: CREATE TABLES ===")
create_tables()
print()

print("=== STEP 2: FETCH DATA ===")
products_raw = fetch_products()
carts_raw    = fetch_carts()
users_raw    = fetch_users()
print()

print("=== STEP 3: TRANSFORM DATA ===")
df_products, df_reviews  = transform_products(products_raw)
df_carts, df_cart_items  = transform_carts(carts_raw)
df_users                 = transform_users(users_raw)
print()

print("=== STEP 4: WRITE TO SQL SERVER ===")
write_products(df_products, df_reviews)
write_carts(df_carts, df_cart_items)
write_users(df_users)
print()

print("=== ALL STEPS COMPLETE ===")

=== STEP 1: CREATE TABLES ===
SUCCESS: Connected to localhost\SQLEXPRESS - EcommerceDB
SUCCESS: products table ready
SUCCESS: product_reviews table ready
SUCCESS: carts table ready
SUCCESS: cart_items table ready
SUCCESS: users table ready

=== STEP 2: FETCH DATA ===
SUCCESS: Fetched 100 records from https://dummyjson.com/products?limit=100
SUCCESS: Fetched 100 records from https://dummyjson.com/carts?limit=100
SUCCESS: Fetched 100 records from https://dummyjson.com/users?limit=100

=== STEP 3: TRANSFORM DATA ===
SUCCESS: Transformed products  - 100 rows, 300 reviews
SUCCESS: Transformed carts     - 100 rows, 378 line items
SUCCESS: Transformed users     - 100 rows

=== STEP 4: WRITE TO SQL SERVER ===
SUCCESS: Connected to localhost\SQLEXPRESS - EcommerceDB
SUCCESS: SQLAlchemy engine connected to localhost\SQLEXPRESS - EcommerceDB
SUCCESS: Inserted 100 rows into products
SUCCESS: Inserted 300 rows into product_reviews
SUCCESS: Connected to localhost\SQLEXPRESS - EcommerceDB
SUCCESS: SQ

In [23]:
import importlib
import pipeline
importlib.reload(pipeline)

from pipeline import run_pipeline

# Run the entire ETL pipeline end to end
run_pipeline()

  ECOMMERCE ETL PIPELINE - STARTING

STEP 1: Creating tables in SQL Server...
SUCCESS: Connected to localhost\SQLEXPRESS - EcommerceDB
SUCCESS: products table ready
SUCCESS: product_reviews table ready
SUCCESS: carts table ready
SUCCESS: cart_items table ready
SUCCESS: users table ready

STEP 2: Extracting data from DummyJSON API...
SUCCESS: Fetched 100 records from https://dummyjson.com/products?limit=100
SUCCESS: Fetched 100 records from https://dummyjson.com/carts?limit=100
SUCCESS: Fetched 100 records from https://dummyjson.com/users?limit=100

STEP 3: Transforming raw data into DataFrames...
SUCCESS: Transformed products  - 100 rows, 300 reviews
SUCCESS: Transformed carts     - 100 rows, 378 line items
SUCCESS: Transformed users     - 100 rows

STEP 4: Loading data into SQL Server...
SUCCESS: Connected to localhost\SQLEXPRESS - EcommerceDB
SUCCESS: SQLAlchemy engine connected to localhost\SQLEXPRESS - EcommerceDB
SUCCESS: Inserted 100 rows into products
SUCCESS: Inserted 300 rows 

True